# SHL Hiring Assessment 2026: Rubric-Driven Spoken Grammar Scoring Engine

**Goal:** predict a 0-5 grammar score from a 45-60 s speech clip (Pearson r and RMSE).

## Approach (report)
The engine is built directly from the grading rubric. No model is fine-tuned; pretrained models are used only as frozen feature extractors.

| Rubric criterion | Feature group | How it is measured |
|---|---|---|
| Sentence structure / incomplete sentences (1-2) | `structure` | share of fragments (no verb), clauses without subject, sentence count and length (spaCy) |
| Control of complex grammar (4-5) | `complexity` | dependency-tree depth, subordinate/relative clauses, coordination, tense variety, modals, aspect |
| Grammatical accuracy, errors (2-4) | `accuracy` | CoLA grammatical-acceptability per sentence, GPT-2 surprisal |
| Self-correction, fluency (4-5) | `fluency` | fillers, repeats, restarts, pauses, speech rate (Whisper word timestamps) |
| Intelligibility of the structure | `asr_conf` | Whisper word-level confidence |

**Preprocessing (all clips):** mono, 16 kHz, edge-silence trim, peak normalisation, followed by a quality-check plot.

Pipeline: preprocessed audio -> Whisper transcript + timings -> rubric features -> Ridge and LightGBM. Frozen-embedding models add context: DeBERTa on text, and wav2vec2, WavLM and HuBERT on audio (WavLM and HuBERT are the strongest single models), and a non-negative Ridge stacker blends everything on out-of-fold predictions. Validation is 5-fold stratified CV with identical folds for all models; predictions are clipped to [0, 5].
A feature-group ablation shows which rubric criterion contributes most.

In [ ]:
!pip -q install faster-whisper lightgbm spacy
!python -m spacy download en_core_web_sm -q
import os, glob, re, pickle, warnings
import numpy as np, pandas as pd, torch, librosa
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error
warnings.filterwarnings('ignore')

SEED = 42
dev = 'cuda'
root = glob.glob('/kaggle/input/**/Dataset_Final', recursive=True)[0]
train = pd.read_csv(f'{root}/train.csv')
test  = pd.read_csv(f'{root}/test.csv')
sub   = pd.read_csv(f'{root}/sample_submission.csv')
y = train.label.values.astype(float)
def rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))

# ---------- audio preprocessing (used by every downstream model) ----------
def load_audio(path, sr=16000, top_db=30):
    # 1) decode to mono at 16 kHz (Whisper / wav2vec2 expect this)
    a, _ = librosa.load(path, sr=sr, mono=True)
    # 2) trim leading/trailing silence so pauses at the edges do not distort speech-rate features
    a, _ = librosa.effects.trim(a, top_db=top_db)
    # 3) peak-normalise to remove recording-level differences between speakers/devices
    peak = np.abs(a).max()
    return (a / peak).astype(np.float32) if peak > 0 else a.astype(np.float32)
print('csv shapes  train', train.shape, '| test.csv', test.shape, '| sample_submission', sub.shape)
print('sample_submission ids found in test.csv:', len(set(sub.filename) & set(test.filename)), 'of', len(sub))

# File names are NOT unique across splits (212 of 216 test names also exist in the train folder as different
# recordings), so every lookup below is keyed by (split, name) and uses explicit paths.
test['path'] = [f'{root}/test/{f}' for f in test.filename]
assert all(os.path.exists(p) for p in test.path), 'test.csv lists files missing from the test folder'
print('test.csv files found on disk:', len(test))

## 1. Data overview

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
train.label.value_counts().sort_index().plot.bar(ax=ax[0], title='Label distribution')
durs = [librosa.get_duration(path=f'{root}/train/{f}') for f in train.filename[:100]]
ax[1].hist(durs, bins=20); ax[1].set_title('Clip duration (s), 100 samples')
plt.tight_layout(); plt.show()

## 1b. Audio preprocessing and quality check
Every clip goes through the same steps before any model sees it: **decode to mono 16 kHz -> trim edge silence -> peak-normalise**. Trimming matters because speech rate and pause features should reflect the speaker, not the recording start/end. The check below confirms the effect on a sample of clips.

In [ ]:
rows = []
for f in train.filename.sample(150, random_state=SEED):
    raw, _ = librosa.load(f'{root}/train/{f}', sr=16000, mono=True)
    pre = load_audio(f'{root}/train/{f}')
    rows.append(dict(file=f, raw_s=len(raw) / 16000, clean_s=len(pre) / 16000,
                     raw_peak=np.abs(raw).max(), clean_peak=np.abs(pre).max(),
                     raw_rms=np.sqrt((raw ** 2).mean()), clean_rms=np.sqrt((pre ** 2).mean())))
qc = pd.DataFrame(rows)
print(qc.describe().loc[['mean', 'min', 'max']].round(3))
print('clips with < 40 s of speech after trimming:', int((qc.clean_s < 40).sum()))
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
ax[0].hist(qc.raw_s, bins=20, alpha=.6, label='raw'); ax[0].hist(qc.clean_s, bins=20, alpha=.6, label='trimmed'); ax[0].legend(); ax[0].set_title('Duration (s)')
ax[1].hist(qc.raw_rms, bins=20, alpha=.6, label='raw'); ax[1].hist(qc.clean_rms, bins=20, alpha=.6, label='normalised'); ax[1].legend(); ax[1].set_title('RMS level')
f0 = qc.file.iloc[0]; w = load_audio(f'{root}/train/{f0}')
ax[2].plot(np.arange(len(w)) / 16000, w, lw=.3); ax[2].set_title(f'Preprocessed waveform: {f0}')
plt.tight_layout(); plt.show()

## 2. ASR with Whisper: transcript + confidence + pause statistics
Cached to disk, so it only runs once.

In [ ]:
from faster_whisper import WhisperModel
CACHE = '/kaggle/working/asr_cache.pkl'

def asr_one(wm, path):
    audio = load_audio(path)                                   # preprocessed numpy input (also avoids the av decoder)
    dur = len(audio) / 16000
    segs, _ = wm.transcribe(audio, beam_size=5, language='en', word_timestamps=True,
                            condition_on_previous_text=False)
    segs = list(segs)
    words = [w for s in segs for w in s.words]
    text = ' '.join(s.text.strip() for s in segs)
    d = dict(text=text, dur=dur, n_words=len(words))
    if len(words) < 2:
        return d
    p = np.array([w.probability for w in words])
    st = np.array([w.start for w in words]); en = np.array([w.end for w in words])
    gap = np.clip(st[1:] - en[:-1], 0, None)
    d.update(wps=len(words) / dur,
             prob_mean=p.mean(), prob_min=p.min(), prob_p10=np.percentile(p, 10), prob_low=(p < 0.5).mean(),
             seg_logprob=np.mean([s.avg_logprob for s in segs]), seg_nospeech=np.mean([s.no_speech_prob for s in segs]),
             gap_mean=gap.mean(), gap_max=gap.max(), gap_long=(gap > 0.5).sum(), gap_frac=gap.sum() / dur,
             wdur_mean=(en - st).mean(), wdur_std=(en - st).std(),
             speech_span=(en[-1] - st[0]) / dur)
    return d

asr = pickle.load(open(CACHE, 'rb')) if os.path.exists(CACHE) else {}
jobs = [(('train', f), f'{root}/train/{f}') for f in train.filename] +        [(('test', f), p) for f, p in zip(test.filename, test.path)]
todo = [(k, p) for k, p in jobs if k not in asr]          # only transcribe what is not cached yet
print('to transcribe:', len(todo), 'of', len(jobs))
if todo:
    wm = WhisperModel('medium.en', device='cuda', compute_type='float16')
    for k, p in tqdm(todo, desc='whisper'):
        asr[k] = asr_one(wm, p)
    pickle.dump(asr, open(CACHE, 'wb'))
    del wm; torch.cuda.empty_cache()

A_tr = pd.DataFrame([asr[('train', f)] for f in train.filename])
A_te = pd.DataFrame([asr[('test', f)] for f in test.filename])
train['text'], test['text'] = A_tr.text.fillna(''), A_te.text.fillna('')
num_cols = [c for c in A_tr.columns if c != 'text']
A_tr, A_te = A_tr[num_cols], A_te[num_cols].reindex(columns=num_cols)
print(train.text.str.split().str.len().describe())
print(train[['label', 'text']].sample(3, random_state=0).values)

## 3. Rubric-driven features
Each feature group maps to one rubric criterion (see table above). spaCy gives syntax (fragments, tree depth, clauses), a CoLA model gives per-sentence grammaticality, GPT-2 gives surprisal, and Whisper timings give fluency.

In [ ]:
import spacy
from transformers import (AutoTokenizer, AutoModel, AutoModelForCausalLM, AutoModelForSequenceClassification)
nlp = spacy.load('en_core_web_sm', disable=['ner', 'lemmatizer'])

# ---------- structure + complexity (spaCy dependency parse) ----------
def tree_depth(tok):
    # compare token indices, NOT objects: spaCy builds a new Token wrapper on each access,
    # so `tok.head is tok` is never True and the loop would never stop at the root
    d = 0
    while tok.head.i != tok.i and d < 100: tok = tok.head; d += 1
    return d

SYN_NAMES = ['frag_rate', 'nosubj_rate', 'n_sents', 'sent_len', 'depth_mean', 'depth_max',
             'subord_rate', 'coord_rate', 'n_tenses', 'pos_diversity', 'modal_rate', 'aspect_rate', 'aux_rate']
def syn_feats_doc(doc):
    if len(doc) == 0: return [0.0] * len(SYN_NAMES)      # empty transcript guard
    sents = [s for s in doc.sents if any(w.is_alpha for w in s)] or [doc]
    n = len(sents); nw = max(sum(w.is_alpha for w in doc), 1)
    frag = nosubj = 0; depths = []
    for s in sents:
        has_verb = any(w.pos_ in ('VERB', 'AUX') for w in s)
        has_subj = any(w.dep_ in ('nsubj', 'nsubjpass', 'expl', 'csubj') for w in s)
        frag += (not has_verb); nosubj += (has_verb and not has_subj)     # incomplete sentences
        depths.append(max(tree_depth(w) for w in s))
    subord = sum(w.dep_ in ('advcl', 'ccomp', 'xcomp', 'relcl', 'acl', 'csubj') for w in doc)
    coord  = sum(w.dep_ == 'conj' for w in doc)
    tenses = {m for w in doc for m in w.morph.get('Tense')}
    return [frag / n, nosubj / n, n, nw / n, np.mean(depths), np.max(depths), subord / n, coord / n, len(tenses),
            len({w.pos_ for w in doc}), sum(w.tag_ == 'MD' for w in doc) / nw,
            sum(w.tag_ in ('VBG', 'VBN') for w in doc) / nw, sum(w.dep_ in ('aux', 'auxpass') for w in doc) / nw]

# ---------- accuracy: CoLA acceptability and GPT-2 surprisal ----------
@torch.no_grad()
def lm_feats(texts):
    tk = AutoTokenizer.from_pretrained('gpt2'); m = AutoModelForCausalLM.from_pretrained('gpt2').to(dev).eval()
    out = []
    for t in tqdm(texts, desc='gpt2'):
        ids = tk(t, return_tensors='pt', truncation=True, max_length=1024).input_ids.to(dev)
        if ids.shape[1] < 2: out.append([0, 0, 0]); continue
        nll = torch.nn.functional.cross_entropy(m(ids).logits[0, :-1], ids[0, 1:], reduction='none').cpu().numpy()
        out.append([nll.mean(), np.median(nll), (nll > 6).mean()])
    del m; torch.cuda.empty_cache()
    return np.array(out)

@torch.no_grad()
def cola_feats(texts):
    name = 'textattack/roberta-base-CoLA'
    tk = AutoTokenizer.from_pretrained(name); m = AutoModelForSequenceClassification.from_pretrained(name).to(dev).eval()
    out = []
    for t in tqdm(texts, desc='cola'):
        sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', t) if len(s.split()) >= 3] or [t]
        b = tk(sents, return_tensors='pt', padding=True, truncation=True, max_length=128).to(dev)
        p = torch.softmax(m(**b).logits, -1)[:, 1].cpu().numpy()
        out.append([p.mean(), p.min(), p.std(), (p < 0.5).mean()])
    del m; torch.cuda.empty_cache()
    return np.array(out)

# ---------- fluency / self-correction (text part) ----------
FILL = r"\b(um+|uh+|er+|ah+|hmm+|you know|i mean)\b"
REPAIR = r"\b(i mean|sorry|no wait|rather|or rather)\b"
LEX_NAMES = ['ttr', 'filler', 'repeat', 'repeat_bigram', 'repair', 'word_len', 'connect']
def lex_feats(t):
    w = t.lower().split(); n = max(len(w), 1)
    bg = list(zip(w, w[1:]))
    return [len(set(w)) / n, len(re.findall(FILL, t.lower())) / n, sum(a == b for a, b in zip(w, w[1:])) / n,
            (len(bg) - len(set(bg))) / max(len(bg), 1), len(re.findall(REPAIR, t.lower())) / n,
            sum(len(x) for x in w) / n,
            sum(x in ('and', 'but', 'so', 'because', 'which', 'that', 'if', 'when', 'although') for x in w) / n]

LM_NAMES, COLA_NAMES = ['gpt2_nll_mean', 'gpt2_nll_med', 'gpt2_nll_hi'], ['cola_mean', 'cola_min', 'cola_std', 'cola_bad']

def build(df, A):
    texts = df.text.tolist()
    return pd.concat([A.reset_index(drop=True),
                      pd.DataFrame([syn_feats_doc(d) for d in tqdm(nlp.pipe([t[:5000] for t in texts], batch_size=32), total=len(texts), desc='spacy')], columns=SYN_NAMES),
                      pd.DataFrame(lm_feats(texts), columns=LM_NAMES),
                      pd.DataFrame(cola_feats(texts), columns=COLA_NAMES),
                      pd.DataFrame([lex_feats(t) for t in texts], columns=LEX_NAMES)], axis=1)

F_tr, F_te = build(train, A_tr), build(test, A_te)
F_te = F_te[F_tr.columns]
F_tr, F_te = F_tr.fillna(F_tr.median()), F_te.fillna(F_tr.median())

# rubric criterion -> feature group
GROUPS = {
  'structure':  ['frag_rate', 'nosubj_rate', 'n_sents', 'sent_len'],
  'complexity': ['depth_mean', 'depth_max', 'subord_rate', 'coord_rate', 'n_tenses', 'pos_diversity',
                 'modal_rate', 'aspect_rate', 'aux_rate', 'connect', 'ttr', 'word_len'],
  'accuracy':   COLA_NAMES + LM_NAMES,
  'fluency':    ['filler', 'repeat', 'repeat_bigram', 'repair', 'wps', 'gap_mean', 'gap_max', 'gap_long',
                 'gap_frac', 'wdur_mean', 'wdur_std', 'speech_span', 'n_words'],
  'asr_conf':   ['prob_mean', 'prob_min', 'prob_p10', 'prob_low', 'seg_logprob', 'seg_nospeech'],
}
GROUPS = {g: [c for c in cs if c in F_tr.columns] for g, cs in GROUPS.items()}
print(F_tr.shape, {g: len(c) for g, c in GROUPS.items()})
print('corr(cola_mean, label) =', pearsonr(F_tr.cola_mean, y)[0], '(should be positive)')
corr = F_tr.apply(lambda c: pearsonr(c, y)[0]).sort_values()
corr.plot.barh(figsize=(6, 10), title='Pearson r of each feature with label'); plt.show()

## 4. Embeddings: wav2vec2 (audio) and frozen DeBERTa-v3 (transcript)

In [ ]:
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

@torch.no_grad()
def audio_emb(paths):
    nm = 'facebook/wav2vec2-base-960h'
    fe = Wav2Vec2FeatureExtractor.from_pretrained(nm); m = Wav2Vec2Model.from_pretrained(nm).to(dev).eval().half()
    out = []
    for p in tqdm(paths, desc='wav2vec2'):
        a = load_audio(p)
        x = fe(a, sampling_rate=16000, return_tensors='pt').input_values.to(dev).half()
        hs = m(x, output_hidden_states=True).hidden_states
        out.append(np.concatenate([hs[6][0].float().mean(0).cpu().numpy(), hs[12][0].float().mean(0).cpu().numpy()]))
    del m; torch.cuda.empty_cache()
    return np.vstack(out)

FT_NAME = 'microsoft/deberta-v3-base'
tok = AutoTokenizer.from_pretrained(FT_NAME)

@torch.no_grad()
def text_emb(texts, bs=16):
    enc = AutoModel.from_pretrained(FT_NAME).to(dev).eval().half()
    out = []
    for i in range(0, len(texts), bs):
        b = tok(texts[i:i + bs], padding=True, truncation=True, max_length=512, return_tensors='pt').to(dev)
        h = enc(**b).last_hidden_state.float(); m = b['attention_mask'].unsqueeze(-1)
        out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())
    del enc; torch.cuda.empty_cache()
    return np.vstack(out)

AUD_tr = audio_emb([f'{root}/train/{f}' for f in train.filename])
AUD_te = audio_emb(list(test.path))
E_tr, E_te = text_emb(train.text.tolist()), text_emb(test.text.tolist())
print(AUD_tr.shape, E_tr.shape)

## 5. Cross-validation setup and classical models
All models share identical stratified folds, so their out-of-fold (OOF) predictions can be stacked without leakage.

In [ ]:
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.svm import SVR
import lightgbm as lgb

ybin = np.digitize(y, [2, 3, 4])
folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(train, ybin))

def cv_sklearn(make, X, Xt):
    oof, tp = np.zeros(len(y)), np.zeros(len(Xt))
    for tr, va in folds:
        m = make().fit(X[tr], y[tr])
        oof[va] = m.predict(X[va]); tp += m.predict(Xt) / len(folds)
    return oof, tp

ridge = lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(1, 4.5, 14)))
lgbm  = lambda: lgb.LGBMRegressor(n_estimators=500, learning_rate=0.02, num_leaves=8, min_child_samples=15,
                                  subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=2,
                                  verbose=-1, random_state=SEED)
P = {}
P['ridge_text_emb']  = cv_sklearn(ridge, E_tr, E_te)
P['ridge_audio_emb'] = cv_sklearn(ridge, AUD_tr, AUD_te)
P['ridge_feats']     = cv_sklearn(ridge, F_tr.values, F_te.values)
P['lgbm_feats']      = cv_sklearn(lgbm, F_tr.values, F_te.values)
for k, (o, _) in P.items():
    print(f'{k:18s} CV RMSE {rmse(y, np.clip(o, 0, 5)):.3f}  Pearson {pearsonr(y, o)[0]:.3f}')

### Rubric-group ablation
Which rubric criterion carries the signal? Ridge CV with each feature group alone, and with each group left out.

In [ ]:
rows = []
allc = list(F_tr.columns)
base_o, _ = cv_sklearn(ridge, F_tr.values, F_te.values)
rows.append(['ALL', rmse(y, np.clip(base_o, 0, 5)), pearsonr(y, base_o)[0], np.nan])
for g, cs in GROUPS.items():
    alone, _ = cv_sklearn(ridge, F_tr[cs].values, F_te[cs].values)
    rest = [c for c in allc if c not in cs]
    wo, _ = cv_sklearn(ridge, F_tr[rest].values, F_te[rest].values)
    rows.append([g, rmse(y, np.clip(alone, 0, 5)), pearsonr(y, alone)[0], rmse(y, np.clip(wo, 0, 5))])
abl = pd.DataFrame(rows, columns=['group', 'rmse_alone', 'pearson_alone', 'rmse_without']).round(4)
print(abl.to_string(index=False))
abl.set_index('group')[['rmse_alone', 'rmse_without']].plot.bar(figsize=(7, 3.5), title='Ridge CV RMSE: group alone vs group removed'); plt.show()

## 6b. Extra audio models: WavLM and HuBERT
Audio embeddings were the strongest signal (wav2vec2 alone: CV Pearson 0.853), so two more frozen self-supervised speech encoders are added. For each clip, hidden states from layers 4, 8 and 12 are mean- and std-pooled over time (middle layers carry prosody and fluency, later layers carry phonetic content). One Ridge model per encoder uses the same CV folds, and its out-of-fold predictions join the stacker. Nothing is fine-tuned.

In [ ]:
from transformers import AutoFeatureExtractor

@torch.no_grad()
def audio_emb2(name, paths, layers=(4, 8, 12)):
    fe = AutoFeatureExtractor.from_pretrained(name)
    m = AutoModel.from_pretrained(name).to(dev).eval().half()
    out = []
    for p in tqdm(paths, desc=name.split('/')[-1]):
        x = fe(load_audio(p), sampling_rate=16000, return_tensors='pt').input_values.to(dev).half()
        hs = m(x, output_hidden_states=True).hidden_states
        out.append(np.concatenate([g(hs[l][0].float()).cpu().numpy() for l in layers
                                   for g in (lambda h: h.mean(0), lambda h: h.std(0))]))   # mean + std pooling
    del m; torch.cuda.empty_cache()
    return np.vstack(out)

tr_paths = [f'{root}/train/{f}' for f in train.filename]
te_paths = list(test.path)
for tag, name in [('wavlm', 'microsoft/wavlm-base-plus'), ('hubert', 'facebook/hubert-base-ls960')]:
    Xtr, Xte = audio_emb2(name, tr_paths), audio_emb2(name, te_paths)
    P[f'ridge_{tag}'] = cv_sklearn(ridge, Xtr, Xte)
    o = P[f'ridge_{tag}'][0]
    print(f'{tag:8s} CV RMSE {rmse(y, np.clip(o, 0, 5)):.3f}  Pearson {pearsonr(y, o)[0]:.3f}')

## 7. Stacking, evaluation and visualisations

In [ ]:
names = list(P)
O = np.column_stack([P[n][0] for n in names]); T = np.column_stack([P[n][1] for n in names])

meta = Ridge(alpha=1.0, positive=True)
# honest estimate: CV of the stacker over the OOF matrix using a different split
oof_stack = np.clip(cross_val_predict(meta, O, y, cv=KFold(5, shuffle=True, random_state=7)), 0, 5)
meta.fit(O, y)
train_pred = np.clip(meta.predict(O), 0, 5)

res = pd.DataFrame({
    'model': names + ['STACK'],
    'cv_rmse': [rmse(y, np.clip(P[n][0], 0, 5)) for n in names] + [rmse(y, oof_stack)],
    'cv_pearson': [pearsonr(y, P[n][0])[0] for n in names] + [pearsonr(y, oof_stack)[0]]}).round(4)
print(res.to_string(index=False))
print('\nBlend weights:', dict(zip(names, meta.coef_.round(3))), 'intercept', round(meta.intercept_, 3))
print(f'\n>>> TRAIN RMSE (stacker on train set): {rmse(y, train_pred):.4f}')
print(f'>>> TRAIN Pearson:                      {pearsonr(y, train_pred)[0]:.4f}')
print(f'>>> CV (out-of-fold) RMSE:              {rmse(y, oof_stack):.4f}')
print(f'>>> CV (out-of-fold) Pearson:           {pearsonr(y, oof_stack)[0]:.4f}')

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].scatter(y, oof_stack, alpha=0.4); ax[0].plot([0, 5], [0, 5], 'r--')
ax[0].set(xlabel='true', ylabel='predicted (OOF)', title='OOF predictions vs truth')
ax[1].hist(y - oof_stack, bins=30); ax[1].set_title('OOF residuals')
res.set_index('model').cv_rmse.plot.barh(ax=ax[2], title='CV RMSE by model')
plt.tight_layout(); plt.show()

pd.DataFrame({'true': y, 'pred': oof_stack}).groupby(np.round(y)).pred.agg(['mean', 'std', 'count']).plot(
    y='mean', yerr='std', kind='bar', figsize=(6, 3.5), title='Predicted score by true score band (OOF)'); plt.show()

m_imp = lgbm().fit(F_tr.values, y)
pd.Series(m_imp.feature_importances_, index=F_tr.columns).sort_values().tail(15).plot.barh(figsize=(6, 5), title='LightGBM top features'); plt.show()

## 8. Submission

In [ ]:
test_pred = np.clip(meta.predict(T), 0, 5)
out = pd.DataFrame({'filename': test.filename, 'label': test_pred})
out.to_csv('/kaggle/working/submission.csv', index=False)
print(out.shape); print(out.label.describe())
n_match = len(set(sub.filename) & set(out.filename))
print(f'NOTE: only {n_match} of the {len(sub)} sample_submission ids match test.csv ids; '
      'submission.csv follows test.csv (the files actually provided).')

## 9. Summary
Fill in from the evaluation cell before submitting: **Train RMSE = 0.4992, CV RMSE = 0.5036, CV Pearson = 0.9138** (v2 run; v1 without WavLM/HuBERT: CV RMSE 0.574, Pearson 0.886). Leaderboard public score: fill in from Kaggle.

**Why this design:** the rubric grades sentence completeness, syntactic control and complexity, error rate and self-correction. Each is measured by an explicit feature group (table at the top), so every prediction can be traced to rubric-level evidence. The ablation shows which criterion matters most. Frozen DeBERTa/wav2vec2 embeddings add context the hand-built features miss, and the stacker weights all sources using out-of-fold predictions only.

**No fine-tuning:** pretrained models serve only as frozen feature extractors; the scoring logic (features, models, blend) is trained on the 769 labelled clips.

**Limitations / next steps:** Whisper tends to clean up disfluencies, so a verbatim CTC transcript could expose more repairs; an LLM-judged rubric score per transcript is another feature worth testing.